# Scraper by Date Range

Orchestrator that downloads:
- **Squads**: always (may have been updated)
- **Match Stats**: only for matches between `date_start` and `date_end`

In [1]:
import os
import logging
import pandas as pd
from pprint import pprint
from datetime import datetime

In [2]:
from storage_manager import StorageManager
from memory_logger import InMemoryLogHandler
from utils import empty_s3_bucket, fuzzy_intersect_competitions

C:\Users\jbj_0\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Config

In [4]:
# ============================================================
# DATE RANGE FILTER
# Only match stats for matches within this range will be downloaded.
# Format: 'YYYY-MM-DD'
# ============================================================
date_start = '2026-01-01'
date_end   = '2026-04-12'

# ============================================================
# General config (same as main orchestrator)
# ============================================================
restart_from_zero = False
filter_data = False

lst_continents = ["South America"]
lst_countries = ['Argentina']

dict_continent_country_competitions = {
    'South America': {
        'Argentina': [
                      'Torneo Federal A',
                      'Prim B Metro',
                      'Torneo Proyecci\u00f3n'
        ],
    }
}

lst_seasons = ['2025/2026', '2026']

In [5]:
s3_bucket_name = None
dir_schema = "schema"
dir_log = "logs"
log_level = "INFO"
storage_type = 'local'

## Logger

In [6]:
storage_manager = StorageManager(storage_type=storage_type, s3_bucket=None)

log_handler = InMemoryLogHandler()
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s')
log_handler.setFormatter(formatter)
logging.getLogger().addHandler(log_handler)
logging.getLogger().setLevel(getattr(logging, log_level.upper(), logging.INFO))

## Competitions

In [7]:
from scraper_competitions import scrape_and_save_competitions, load_existing_competitions
from scraper_competitions import fetch_country_competitions_dict

current_stage = 'competitions'

dict_country_competitions = fetch_country_competitions_dict(
    storage_type=storage_type,
    s3_bucket=s3_bucket_name,
    schema_dir=dir_schema,
    log_dir=dir_log,
    log_level=log_level,
    verbose=True,
    continents=lst_continents,
    countries=lst_countries
)

dict_continent_country_competitions_requested = fuzzy_intersect_competitions(
    dict_continent_country_competitions,
    dict_country_competitions,
    fuzzy_threshold=80
)

countries_competitions = {
    country: [comp[0] for comp in competitions]
    for continent in dict_continent_country_competitions_requested.values()
    for country, competitions in continent.items()
}
print("Countries and competitions to scrape:")
pprint(countries_competitions)

[ScraperCompetition] Logging to logs\scraper.log at level INFO
[ScraperCompetition] INFO: Initialized ScraperCompetition with storage_type=local, data_dir=data, schema_dir=schema, log_dir=logs
[ScraperCompetition] INFO: Initialized ScraperCompetition with storage: LOCAL
[ScraperCompetition] INFO:    - Filters - Continents: ['South America'], Countries: ['Argentina'], Competitions: None
[ScraperCompetition] INFO: fetch_competitions_data - Fetching competition data...
[ScraperCompetition] INFO: Peticion: https://www.scoresway.com/en_GB/soccer/competitions
[ScraperCompetition] INFO: Headers: {'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7', 'accept-language': 'es-ES,es;q=0.9', 'priority': 'u=0, i', 'referer': 'https://www.scoresway.com/en_GB/soccer/competitions', 'cache-control': 'no-cache', 'pragma': 'no-cache', 'sec-ch-ua': '"Not:A-Brand";v="99", "Google Chrome";v="145", "Chromium";v="145

In [8]:
df_competitions, summary_competitions = scrape_and_save_competitions(
    save_csv=True,
    create_dirs=True,
    storage_type=storage_type,
    s3_bucket=s3_bucket_name,
    schema_dir=dir_schema,
    log_dir=dir_log,
    log_level=log_level,
    verbose=True,
    continents=lst_continents,
    countries=lst_countries,
    competitions_dict=dict_continent_country_competitions_requested
)
print(f"Competitions loaded: {len(df_competitions)}")

timestamp_suffix = datetime.now().strftime('%Y%m%d%H%M%S')
log_handler.dump_to_file(storage_manager, log_dir=dir_log, filename=f'log_{current_stage}_{timestamp_suffix}.log')
log_handler.clear()

[ScraperCompetition] Logging to logs\scraper.log at level INFO
[ScraperCompetition] INFO: Initialized ScraperCompetition with storage_type=local, data_dir=data, schema_dir=schema, log_dir=logs
[ScraperCompetition] INFO: Initialized ScraperCompetition with storage: LOCAL
[ScraperCompetition] INFO:    - Filters - Continents: ['South America'], Countries: ['Argentina'], Competitions: {'South America': {'Argentina': [('Torneo Federal A', 'a08z7wsz0uqx90wsj33g65fcd'), ('Prim B Metro', '66nxwll8fur0rtp9q7wqaki1l'), ('Torneo Proyección', '2f3fydvyjz2svhlc1eo3qflwd')]}}
[ScraperCompetition] INFO: fetch_competitions_data - Fetching competition data...
[ScraperCompetition] INFO: Peticion: https://www.scoresway.com/en_GB/soccer/competitions
[ScraperCompetition] INFO: Headers: {'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7', 'accept-language': 'es-ES,es;q=0.9', 'priority': 'u=0, i', 'referer': 'htt

## Seasons

In [9]:
from scraper_seasons import smart_scrape_seasons

current_stage = 'seasons'

df_seasons, summary_seasons = smart_scrape_seasons(
    df_competitions=df_competitions,
    restart_from_zero=restart_from_zero,
    batch_size=None,
    storage_type=storage_type,
    s3_bucket=s3_bucket_name,
    countries=lst_countries,
    fuzzy_threshold=80,
    seasons=lst_seasons,
    create_dirs=True
)
print(f"Seasons loaded: {len(df_seasons)}")

timestamp_suffix = datetime.now().strftime('%Y%m%d%H%M%S')
log_handler.dump_to_file(storage_manager, log_dir=dir_log, filename=f'log_{current_stage}_{timestamp_suffix}.log')
log_handler.clear()

[ScraperSeason] Logging to logs\scraper.log at level INFO
[ScraperSeason] INFO: Initialized ScraperSeason with storage_type=local, data_dir=data, schema_dir=schema, log_dir=logs
[ScraperSeason] INFO: Web session initialized.
[ScraperSeason] INFO: Filtering by countries: Argentina | Original: 3 | Filtered: 3
[ScraperSeason] WARNING: No previous data or error loading: File not found: schema\all_seasons.csv
[ScraperSeason] INFO: Processing 3 competitions...
[ScraperSeason] INFO: Starting season scraping: 3 competitions, range 0 to 2
[ScraperSeason] INFO: Processing 1/3: Torneo Federal A
[ScraperSeason] Sleeping for 2.00 seconds...
[ScraperSeason] INFO: Found 2 seasons for Torneo Federal A (from selector)
[ScraperSeason] Sleeping for 1.80 seconds...
[ScraperSeason] INFO: Processing 2/3: Prim B Metro
[ScraperSeason] Sleeping for 2.18 seconds...
[ScraperSeason] INFO: Found 2 seasons for Prim B Metro (from selector)
[ScraperSeason] Sleeping for 1.35 seconds...
[ScraperSeason] INFO: Processing

## Fixtures

In [10]:
from scraper_fixture import smart_download_fixtures, load_existing_fixtures

current_stage = 'fixtures'

smart_download_fixtures(
    df_seasons=df_seasons,
    continent=None,
    country=None,
    restart_from_zero=restart_from_zero,
    batch_size=None,
    storage_type=storage_type,
    s3_bucket=s3_bucket_name
)
df_fixtures = load_existing_fixtures(
    storage_type=storage_type,
    s3_bucket=s3_bucket_name
)
print(f"Fixtures loaded: {len(df_fixtures)}")

timestamp_suffix = datetime.now().strftime('%Y%m%d%H%M%S')
log_handler.dump_to_file(storage_manager, log_dir=dir_log, filename=f'log_{current_stage}_{timestamp_suffix}.log')
log_handler.clear()

[ScraperFixture] Logging to logs\scraper.log at level INFO
[ScraperFixture] INFO: Initialized ScraperFixture with storage_type=local, data_dir=data, schema_dir=schema, log_dir=logs
🎯 Applied filters: None
📋 Seasons to process: 3
[ScraperFixture] Logging to logs\scraper.log at level INFO
[ScraperFixture] INFO: Initialized ScraperFixture with storage_type=local, data_dir=data, schema_dir=schema, log_dir=logs
[ScraperFixture] INFO: Resume point found at index 0: Torneo Federal A - 2026
🚀 Starting download from season 1/3
[ScraperFixture] Logging to logs\scraper.log at level INFO
[ScraperFixture] INFO: Initialized ScraperFixture with storage_type=local, data_dir=data, schema_dir=schema, log_dir=logs
[ScraperFixture] INFO: Starting fixture download: 3 seasons
[ScraperFixture] INFO: Processing 1/3: Torneo Federal A - 2026
[ScraperFixture] INFO: Requesting API URL: https://api.performfeeds.com/soccerdata/match/ft1tiv1inq7v1sk3y9tv12yh5/?_rt=c&tmcl=1q9caek968o8ubc43inhg6uxg&live=yes&_pgSz=400&

## Matches From Fixtures

In [11]:
from processor_fixtures import process_matches_by_filters

current_stage = 'list_matches'

df_matches = process_matches_by_filters(
    df_seasons=df_seasons,
    continent=None,
    country=None,
    save_consolidated=True,
    consolidated_filename='all_matches.csv',
    save_individual=True,
    storage_type=storage_type,
    s3_bucket=s3_bucket_name
)
print(f"Total matches extracted: {len(df_matches)}")

timestamp_suffix = datetime.now().strftime('%Y%m%d%H%M%S')
log_handler.dump_to_file(storage_manager, log_dir=dir_log, filename=f'log_{current_stage}_{timestamp_suffix}.log')
log_handler.clear()

[ProcessorFixture] Logging to logs\scraper.log at level INFO
[ProcessorFixture] INFO: Initialized ProcessorFixture with storage_type=local, data_dir=data, schema_dir=schema, log_dir=logs
[ProcessorFixture] INFO: Starting processing of 3 seasons
[ProcessorFixture] INFO: Processed fixture: data\south_america\argentina\torneo_federal_a\a08z7wsz0uqx90wsj33g65fcd\2026\fixture.json → 306 matches
[ProcessorFixture] INFO: Saved individual matches: data\south_america\argentina\torneo_federal_a\a08z7wsz0uqx90wsj33g65fcd\2026\matches.csv
[ProcessorFixture] INFO: Processed fixture: data\south_america\argentina\prim_b_metro\66nxwll8fur0rtp9q7wqaki1l\2026\fixture.json → 400 matches
[ProcessorFixture] INFO: Saved individual matches: data\south_america\argentina\prim_b_metro\66nxwll8fur0rtp9q7wqaki1l\2026\matches.csv
[ProcessorFixture] INFO: Processed fixture: data\south_america\argentina\torneo_proyeccion\2f3fydvyjz2svhlc1eo3qflwd\2026\fixture.json → 324 matches
[ProcessorFixture] INFO: Saved individ

c:\Users\jbj_0\GitHub\idvu23_updated\scraping\processor_fixtures.py:273: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df_matches["date"] = pd.to_datetime(df_matches["date"], errors="ignore")
c:\Users\jbj_0\GitHub\idvu23_updated\scraping\utils.py:427: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_extended = pd.concat([df_existing, pd.DataFrame(new_rows, columns=df_new.columns)], ignore_index=True)


## Filter Matches by Date Range

In [12]:
# Convert the date column to datetime for comparison
df_matches['date'] = pd.to_datetime(df_matches['date'], errors='coerce')

dt_start = pd.to_datetime(date_start)
dt_end = pd.to_datetime(date_end)

mask = (df_matches['date'] >= dt_start) & (df_matches['date'] <= dt_end)
df_matches_filtered = df_matches[mask].reset_index(drop=True)

print(f"Matches in date range [{date_start} -> {date_end}]: {len(df_matches_filtered)} out of {len(df_matches)}")
if not df_matches_filtered.empty:
    print(f"Date range in filtered data: {df_matches_filtered['date'].min().date()} to {df_matches_filtered['date'].max().date()}")
    display(df_matches_filtered.head(10))

# === DEBUG: check columns and normalize_match_row output ===
from utils import normalize_match_row
print(f"\nColumns in df_matches_filtered: {list(df_matches_filtered.columns)}")
if not df_matches_filtered.empty:
    sample = df_matches_filtered.iloc[0]
    norm = normalize_match_row(sample)
    print(f"\nSample row normalized:")
    for k, v in norm.items():
        flag = "  <<< MISSING!" if not v and k in ['match_id','continent','country','competition','competition_id','season','tournament_id'] else ""
        print(f"  {k}: {repr(v)}{flag}")

Matches in date range [2026-01-01 -> 2026-04-12]: 248 out of 1030
Date range in filtered data: 2026-02-03 to 2026-04-12


,match_id,date,date_raw,time,home_team,away_team,home_score,away_score,venue,status,coverage,last_updated,tournament_id,competition,competition_id,country,continent,season,fixture_path,attendance,weather_temperature,weather_conditions
0,b2lqx30f3wps420hi72nzsb2s,2026-04-12,2026-04-12Z,22:00:00Z,Sarmiento de La Banda,Juventud Antoniana,None,None,Estadio Ciudad de La Banda,None,7,2026-04-08T23:52:09Z,1q9caek968o8ubc43inhg6uxg,Torneo Federal A,a08z7wsz0uqx90wsj33g65fcd,Argentina,South America,2026,data\south_america\argentina\torneo_federal_a\...,None,None,None
1,59l6cvawf5hq3r17swwpsbg2c,2026-04-12,2026-04-12Z,20:00:00Z,9 de Julio Rafaela,Atlético Escobar,None,None,Estadio Germán Malacho Soltermam,None,7,2026-04-08T23:52:32Z,1q9caek968o8ubc43inhg6uxg,Torneo Federal A,a08z7wsz0uqx90wsj33g65fcd,Argentina,South America,2026,data\south_america\argentina\torneo_federal_a\...,None,None,None
2,xceu106ov2ewxzw0mv2190yc,2026-04-12,2026-04-12Z,19:30:00Z,Atenas,Huracán Las Heras,None,None,Estadio 9 de Julio,None,7,2026-04-08T23:54:36Z,1q9caek968o8ubc43inhg6uxg,Torneo Federal A,a08z7wsz0uqx90wsj33g65fcd,Argentina,South America,2026,data\south_america\argentina\torneo_federal_a\...,None,None,None
3,b1343wlb1d453ctdxnqcsmm8k,2026-04-12,2026-04-12Z,19:30:00Z,San Martín Formosa,Boca Unidos,None,None,Estadio 17 de Octubre,None,7,2026-04-08T23:52:33Z,1q9caek968o8ubc43inhg6uxg,Torneo Federal A,a08z7wsz0uqx90wsj33g65fcd,Argentina,South America,2026,data\south_america\argentina\torneo_federal_a\...,None,None,None
4,57kkr3n1h2czixjim8qn72tqs,2026-04-12,2026-04-12Z,19:30:00Z,Douglas Haig,El Linqueño,None,None,Estadio Miguel Morales,None,7,2026-04-08T23:52:32Z,1q9caek968o8ubc43inhg6uxg,Torneo Federal A,a08z7wsz0uqx90wsj33g65fcd,Argentina,South America,2026,data\south_america\argentina\torneo_federal_a\...,None,None,None
5,yrh1d392138z1wcifie0usyc,2026-04-12,2026-04-12Z,19:00:00Z,Argentino Monte Maíz,FADEP,None,None,Estadio Modesto Marrone,None,7,2026-04-08T23:54:36Z,1q9caek968o8ubc43inhg6uxg,Torneo Federal A,a08z7wsz0uqx90wsj33g65fcd,Argentina,South America,2026,data\south_america\argentina\torneo_federal_a\...,None,None,None
6,5926ujesbcaazk2rrlxi9y39w,2026-04-12,2026-04-12Z,19:00:00Z,Gimnasia Chivilcoy,Sportivo Las Parejas,None,None,Estadio José María Paz,None,7,2026-04-08T23:52:32Z,1q9caek968o8ubc43inhg6uxg,Torneo Federal A,a08z7wsz0uqx90wsj33g65fcd,Argentina,South America,2026,data\south_america\argentina\torneo_federal_a\...,None,None,None
7,582eihyulwvuvzwd2biwb22vo,2026-04-12,2026-04-12Z,19:00:00Z,Defensores Belgrano VR,Sportivo Belgrano,None,None,Estadio Salomón Boeseldín,None,7,2026-04-08T23:52:32Z,1q9caek968o8ubc43inhg6uxg,Torneo Federal A,a08z7wsz0uqx90wsj33g65fcd,Argentina,South America,2026,data\south_america\argentina\torneo_federal_a\...,None,None,None
8,ya8y283mpt390iftly98nbis,2026-04-12,2026-04-12Z,18:30:00Z,Juventud Unida Univ.,Deportivo Rincón,None,None,Estadio Mario Sebastián Diez,None,7,2026-04-08T23:54:36Z,1q9caek968o8ubc43inhg6uxg,Torneo Federal A,a08z7wsz0uqx90wsj33g65fcd,Argentina,South America,2026,data\south_america\argentina\torneo_federal_a\...,None,None,None
9,xt6zpze3ptbfa4pwmscso74k,2026-04-12,2026-04-12Z,18:30:00Z,Cipolletti,Costa Brava,None,None,Estadio La Visera de Cemento,None,7,2026-04-08T23:54:36Z,1q9caek968o8ubc43inhg6uxg,Torneo Federal A,a08z7wsz0uqx90wsj33g65fcd,Argentina,South America,2026,data\south_america\argentina\torneo_federal_a\...,None,None,None



Columns in df_matches_filtered: ['match_id', 'date', 'date_raw', 'time', 'home_team', 'away_team', 'home_score', 'away_score', 'venue', 'status', 'coverage', 'last_updated', 'tournament_id', 'competition', 'competition_id', 'country', 'continent', 'season', 'fixture_path', 'attendance', 'weather_temperature', 'weather_conditions']

Sample row normalized:
  match_id: 'b2lqx30f3wps420hi72nzsb2s'
  date: Timestamp('2026-04-12 00:00:00')
  date_raw: '2026-04-12Z'
  time: '22:00:00Z'
  home_team: 'Sarmiento de La Banda'
  away_team: 'Juventud Antoniana'
  home_score: None
  away_score: None
  venue: 'Estadio Ciudad de La Banda'
  status: None
  coverage: '7'
  last_updated: '2026-04-08T23:52:09Z'
  tournament_id: '1q9caek968o8ubc43inhg6uxg'
  competition: 'Torneo Federal A'
  competition_id: 'a08z7wsz0uqx90wsj33g65fcd'
  country: 'Argentina'
  continent: 'South America'
  season: '2026'
  fixture_path: 'data\\south_america\\argentina\\torneo_federal_a\\a08z7wsz0uqx90wsj33g65fcd\\2026\\fixt

## Match Stats (date-filtered)

In [13]:
from scraper_match_stats import ScrapeMatchStats

current_stage = 'matches_stats'

# Use the scraper directly with skip_existing=False to FORCE re-download
# (smart_download_match_stats would skip already-existing files)
scraper_ms = ScrapeMatchStats(storage_type=storage_type, s3_bucket=s3_bucket_name)
scraper_ms.set_delay(1.0)

print(f"Passing {len(df_matches_filtered)} matches to process_matches")
print(f"Columns: {list(df_matches_filtered.columns)}")

df_matches_stats, stats_matches_stats = scraper_ms.process_matches(
    df_matches=df_matches_filtered,  # <-- only matches within date range
    filters=None,
    skip_existing=False,             # <-- FORCE download even if file exists
    start_index=0,
    limit=None,
    save_consolidated=True,
    consolidated_filename='all_matches_stats.csv'
)

print(f"\nMatch stats downloaded: {len(df_matches_stats)}")
print(f"Stats: total={stats_matches_stats.get('total_matches',0)}, "
      f"processed={stats_matches_stats.get('processed',0)}, "
      f"success={stats_matches_stats.get('success',0)}, "
      f"skipped={stats_matches_stats.get('skipped',0)}, "
      f"failures={stats_matches_stats.get('failures',0)}")

timestamp_suffix = datetime.now().strftime('%Y%m%d%H%M%S')
log_handler.dump_to_file(storage_manager, log_dir=dir_log, filename=f'log_{current_stage}_{timestamp_suffix}.log')
log_handler.clear()

[ScrapeMatchStats] Logging to logs\scraper.log at level INFO
[ScrapeMatchStats] INFO: Initialized ScrapeMatchStats with storage_type=local, data_dir=data, schema_dir=schema, log_dir=logs
[ScrapeMatchStats] INFO: Delay set to 1.0 seconds
Passing 248 matches to process_matches
Columns: ['match_id', 'date', 'date_raw', 'time', 'home_team', 'away_team', 'home_score', 'away_score', 'venue', 'status', 'coverage', 'last_updated', 'tournament_id', 'competition', 'competition_id', 'country', 'continent', 'season', 'fixture_path', 'attendance', 'weather_temperature', 'weather_conditions']
[ScrapeMatchStats] INFO: Starting match stats download: 248 matches
[ScrapeMatchStats] INFO: Processing 1/248: Sarmiento de La Banda vs Juventud Antoniana
dir_path:  data\south_america\argentina\torneo_federal_a\a08z7wsz0uqx90wsj33g65fcd\2026\1q9caek968o8ubc43inhg6uxg\matchstats
[DEBUG] Ensuring directory exists: data\south_america\argentina\torneo_federal_a\a08z7wsz0uqx90wsj33g65fcd\2026\1q9caek968o8ubc43inhg6

c:\Users\jbj_0\GitHub\idvu23_updated\scraping\utils.py:427: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_extended = pd.concat([df_existing, pd.DataFrame(new_rows, columns=df_new.columns)], ignore_index=True)


## Squads (always downloaded)

In [14]:
from scraper_squads import download_squads_by_filters, load_existing_squads

current_stage = 'squads'

# Use download_squads_by_filters directly with skip_existing=False
# to FORCE re-download all squads (they may have been updated)
stats_squads = download_squads_by_filters(
    df_seasons=df_seasons,
    continent=None,
    country=None,
    skip_existing=False,             # <-- FORCE download even if file exists
    start_index=0,
    limit=None,
    page_size=50,
    detailed=True,
    storage_type=storage_type,
    s3_bucket=s3_bucket_name,
    save_consolidated=True,
    consolidated_filename='all_squads.csv'
)

df_squads = load_existing_squads(
    storage_type=storage_type,
    s3_bucket=s3_bucket_name
)
print(f"Squads downloaded: {len(df_squads)}")
print(f"Stats: success={stats_squads.get('success',0)}, "
      f"skipped={stats_squads.get('skipped',0)}, "
      f"errors={stats_squads.get('errors',0)}")

timestamp_suffix = datetime.now().strftime('%Y%m%d%H%M%S')
log_handler.dump_to_file(storage_manager, log_dir=dir_log, filename=f'log_{current_stage}_{timestamp_suffix}.log')
log_handler.clear()

[ScraperSquads] Logging to logs\scraper.log at level INFO
[ScraperSquads] INFO: Initialized ScraperSquads with storage_type=local, data_dir=data, schema_dir=schema, log_dir=logs
[ScraperSquads] INFO: Pagination set: 50 per page, page 1, detailed: True
[ScraperSquads] INFO: Starting squads download: 3 seasons
[ScraperSquads] INFO: Processing 1/3: Torneo Federal A - 2026
continent                                              South America
country                                                    Argentina
competition                                         Torneo Federal A
competition_id                             a08z7wsz0uqx90wsj33g65fcd
competition_url    https://www.scoresway.com/en_GB/soccer/torneo-...
season                                                          2026
season_url         https://www.scoresway.com/en_GB/soccer/torneo-...
results_url        https://www.scoresway.com/en_GB/soccer/torneo-...
Name: 0, dtype: object
[ScraperSquads] INFO: Requesting API URL: https://api

## Summary

In [15]:
print('='*60)
print(f'Date range: {date_start} -> {date_end}')
print(f'Competitions: {len(df_competitions)}')
print(f'Seasons: {len(df_seasons)}')
print(f'Total matches found: {len(df_matches)}')
print(f'Matches in date range: {len(df_matches_filtered)}')
print(f'Match stats downloaded: {len(df_matches_stats)}')
print(f'Squads downloaded: {len(df_squads)}')
print('='*60)

Date range: 2026-01-01 -> 2026-04-12
Competitions: 3
Seasons: 3
Total matches found: 1030
Matches in date range: 248
Match stats downloaded: 248
Squads downloaded: 3
